# Занятие 6. Ручные признаки изображений
## От пикселей к компактному описанию

**Часть 1** — демонстрация преподавателя.  
**Часть 2** — самостоятельные операции по 1–2 строки в `# YOUR CODE HERE`.

## Часть 1. Демонстрация преподавателя

Сегодня меняем **representation**: одно и то же изображение будем описывать разными наборами чисел.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

plt.rcParams['figure.figsize'] = (8, 4)
digits = load_digits()
images = digits.images.astype(np.float32)
labels = digits.target
sample = images[21]
print('label:', labels[21], 'shape:', sample.shape)
plt.imshow(sample, cmap='gray', vmin=0, vmax=16); plt.axis('off'); plt.show()

### 1. Сырые пиксели — baseline
Можно просто получить 64 признака. Это допустимо, но не единственное представление.

In [ ]:
raw = sample.ravel()
print('raw features:', raw.shape)

### 2. Несколько глобальных и региональных статистик
Человек заранее решает, какие измерения могут быть полезны.

In [ ]:
h, w = sample.shape
mean_all = sample.mean()
top = sample[:h//2].mean(); bottom = sample[h//2:].mean()
left = sample[:, :w//2].mean(); right = sample[:, w//2:].mean()
center = sample[2:6, 2:6].mean()
print(mean_all, top, bottom, left, right, center)

### 3. Локальные средние по блокам
Разделим 8×8 на шестнадцать блоков 2×2. Это 16 признаков вместо 64.

In [ ]:
def block_means(img):
    return img.reshape(4, 2, 4, 2).mean(axis=(1, 3)).ravel()

blocks = block_means(sample)
print(blocks.shape, blocks.reshape(4, 4))

### 4. Симметрия и сила границ
Это уже не значения конкретных пикселей, а свойства формы.

In [ ]:
symmetry = np.abs(sample - np.fliplr(sample)).mean()
edge_x = np.abs(np.diff(sample, axis=1)).mean()
edge_y = np.abs(np.diff(sample, axis=0)).mean()
print('symmetry:', symmetry, 'edge x/y:', edge_x, edge_y)

### 5. Соберём простой ручной дескриптор
Он содержит девять понятных чисел.

In [ ]:
def manual_features(img):
    h, w = img.shape
    return np.array([
        img.mean(), img[:h//2].mean(), img[h//2:].mean(),
        img[:, :w//2].mean(), img[:, w//2:].mean(),
        img[h//4:3*h//4, w//4:3*w//4].mean(),
        np.abs(img - np.fliplr(img)).mean(),
        np.abs(np.diff(img, axis=1)).mean(),
        np.abs(np.diff(img, axis=0)).mean(),
    ], dtype=np.float32)

print(manual_features(sample))

### 6. Упрощённая идея HOG: локальные направления границ
Это **не полная реализация HOG**, а короткая версия идеи: для каждой из 4 областей считаем гистограмму направлений градиента.

In [ ]:
def orientation_features(img, bins=8):
    a = img.astype(np.float32)
    gx = np.zeros_like(a); gy = np.zeros_like(a)
    gx[:, 1:-1] = (a[:, 2:] - a[:, :-2]) / 2
    gy[1:-1, :] = (a[2:, :] - a[:-2, :]) / 2
    mag = np.hypot(gx, gy)
    ang = (np.arctan2(gy, gx) + np.pi) % np.pi
    feats = []
    for y0 in (0, 4):
        for x0 in (0, 4):
            hist, _ = np.histogram(ang[y0:y0+4, x0:x0+4], bins=bins, range=(0, np.pi), weights=mag[y0:y0+4, x0:x0+4])
            feats.extend(hist / (np.linalg.norm(hist) + 1e-8))
    return np.asarray(feats, dtype=np.float32)

orient = orientation_features(sample)
print('orientation features:', orient.shape)

### 7. Извлекаем признаки для всего датасета
Теперь каждая строка — одно изображение, каждый столбец — один вручную заданный признак.

In [ ]:
X_manual = np.stack([manual_features(img) for img in images])
X_blocks = np.stack([block_means(img) for img in images])
X_orient = np.stack([orientation_features(img) for img in images])
print('raw   :', images.reshape(len(images), -1).shape)
print('manual:', X_manual.shape)
print('blocks:', X_blocks.shape)
print('orient:', X_orient.shape)

---
## Часть 2. Самостоятельные мини-задачи

Работаем с `sample` и уже определёнными функциями.

### Задание 1. Средняя яркость
Запишите её в `mean_s`.

In [ ]:
# YOUR CODE HERE (1 строка)
# mean_s = ...

### Задание 2. Верх и низ
Запишите средние яркости половин в `top_s`, `bottom_s`.

In [ ]:
# YOUR CODE HERE (2 строки)
# top_s = ...
# bottom_s = ...

### Задание 3. Центр
Средняя яркость области `sample[2:6, 2:6]` → `center_s`.

In [ ]:
# YOUR CODE HERE (1 строка)
# center_s = ...

### Задание 4. Симметрия
Средняя абсолютная разница с отражением слева направо → `symmetry_s`.

In [ ]:
# YOUR CODE HERE (1 строка)
# symmetry_s = ...

### Задание 5. Сила горизонтальных перепадов
`edge_x_s` — средняя абсолютная разность соседних столбцов.

In [ ]:
# YOUR CODE HERE (1 строка)
# edge_x_s = ...

### Задание 6. Локальные средние
Получите `blocks_s` через готовую `block_means`.

In [ ]:
# YOUR CODE HERE (1 строка)
# blocks_s = ...

### Задание 7. Направления границ
Получите `orient_s` через `orientation_features`.

In [ ]:
# YOUR CODE HERE (1 строка)
# orient_s = ...

### Задание 8. Матрица ручных признаков
Создайте `X_manual_s` для всех `images`.

In [ ]:
# YOUR CODE HERE (1 строка)
# X_manual_s = ...

### Задание 9. Матрица локальных средних
Создайте `X_blocks_s`.

In [ ]:
# YOUR CODE HERE (1 строка)
# X_blocks_s = ...

### Мини-вопрос
Почему более короткий вектор признаков иногда может работать почти так же хорошо, как все пиксели? И почему это не гарантировано?